# Simple Rubin Injection Demo

This notebook shows the reduced workflow:

1. Configure the synthetic cluster run
2. Load one Rubin image
3. Inject clusters with one pipeline call
4. Run a user detection function (MCI example shown here)
5. Match detections and compute completeness
6. Make and save selected plots from one `make_plots` call

In [ ]:
from pathlib import Path
import sys
import inspect
import importlib

# Robustly resolve the project root that contains src/config.py.
cwd = Path.cwd().resolve()
candidates = [
    cwd,
    cwd.parent,
    cwd / 'star-cluster-injection-pipeline',
    cwd.parent / 'star-cluster-injection-pipeline',
]

repo_root = next((p for p in candidates if (p / 'src' / 'config.py').exists()), None)
if repo_root is None:
    raise RuntimeError(
        f'Could not find project root containing src/config.py from cwd={cwd}'
    )

# Ensure this repo root is first on import path.
sys.path = [p for p in sys.path if p != str(repo_root)]
sys.path.insert(0, str(repo_root))

# Force-clear any previously loaded src modules from another location.
for name in list(sys.modules.keys()):
    if name == 'src' or name.startswith('src.'):
        del sys.modules[name]

from lsst.daf.butler import Butler
from src.config import InjectionConfig, ClusterConfig
from src.pipeline import InjectionPipeline

print('Using repo_root:', repo_root)
print('InjectionConfig loaded from:', inspect.getsourcefile(InjectionConfig))
print('InjectionConfig args:', InjectionConfig.__init__.__code__.co_varnames)

## 1. Configure The Run

If you see an unexpected constructor error, rerun the import cell above first.
It force-reloads local `src` modules to avoid stale kernel imports.

In [ ]:
BUTLER_REPO = 'dp02'
BUTLER_COLLECTIONS = '2.2i/runs/DP0.2'
DATA_ID = {'tract': 3828, 'patch': 24}

# Safety check: confirms this class supports the notebook arguments.
assert 'cutout_size' in InjectionConfig.__init__.__code__.co_varnames, (
    'Loaded InjectionConfig does not include cutout_size. '
    'Rerun the import cell above (or restart kernel and rerun from top).'
)

config = InjectionConfig(
    run_name='simple_rubin_mci_demo',
    band='i',
    cutout_size=1200,
    n_clusters=250,
    seed=42,
    edge_buffer=50,
    use_actual_psf=True,
    psf_fwhm_fallback=3.5,
    output_dir=str(repo_root / 'plots' / 'simple_rubin_mci_demo'),
    cluster_config=ClusterConfig(
        profile_type='king',
        mag_min=20.0,
        mag_max=26.0,
        r_half_min=2.0,
        r_half_max=10.0,
        concentration_min=5.0,
        concentration_max=30.0,
    ),
)

config

## 2. Load One Rubin Image

This loads the coadd cutout and keeps the PSF context inside the pipeline object.

In [ ]:
butler = Butler(BUTLER_REPO, collections=BUTLER_COLLECTIONS)
pipe = InjectionPipeline(config)
pipe.load_data(butler=butler, data_id=DATA_ID)

print(f'Loaded image shape: {pipe.image.shape}')
print(f'Active band: {config.band}')

## 3. Inject Synthetic Clusters

These are now one-line backend calls.

In [ ]:
catalog = pipe.generate_catalog()
injected_image, injection_info = pipe.inject(
    psf_fwhm_fallback=config.psf_fwhm_fallback,
    verbose=True,
)

print(f'Injected clusters: {len(injection_info)}')

## 4. Detection Function (MCI Example For Users)

This cell is intentionally front-end code so users can swap in their own detector.

Required detector output format:
- `detections` must be a `list[dict]`
- every entry must include `x` and `y` pixel coordinates
- optional keys like `flux`, `snr`, `r_half`, `magnitude` unlock extra diagnostics

In [ ]:
from src.detection import run_cluster_detection


def mci_detector(image):
    """Example detector users can replace with their own pipeline."""
    return run_cluster_detection(
        image=image,
        psf_fwhm=config.psf_fwhm_fallback,
        threshold_sigma=3.0,
        mci_max=0.85,
        snr_min=3.0,
        r_half_min=1.0,
        ellipticity_max=0.6,
        box_size=64,
        pixel_scale=config.pixel_scale,
        use_multiscale=True,
        use_mci=True,
        verbose=True,
    )


detections = pipe.detect_with(mci_detector)
print(f'Detections: {len(detections)}')

## 5. Match Detections And Compute Completeness

The same step also gives you the summary statistics for recovery.

In [ ]:
stats = pipe.analyze(match_radius=5.0)
stats

## 6. Make And Save Plots

`make_plots` now handles standard and poster-style outputs from one call.

- `plots` controls which figures are displayed
- saved outputs include completeness, position, summary, PSF histogram, and postage stamps

In [ ]:
results = pipe.make_plots(
    plots=['before_after', 'completeness_1d', 'completeness_2d', 'psf_fwhm_hist'],
    show=True,
    save=True,
    poster_style=False,
    n_stamps=4,
)

print('Output directory:', config.output_dir)
print('Saved files:')
for k, v in results['saved'].items():
    print(f' - {k}: {v}')

## 7. Optional Poster-Style Display

Use the same `make_plots` function with `poster_style=True` for larger text.

In [ ]:
_ = pipe.make_plots(
    plots=['before_after', 'completeness_1d'],
    show=True,
    save=True,
    poster_style=True,
    n_stamps=2,
)

## 8. Optional: Plug In Your Own Detector

Replace only the detector function. The rest of the pipeline stays the same.

In [ ]:
# Example pattern:
# def my_detector(image):
#     return [
#         {'x': 100.0, 'y': 120.0, 'flux': 42.0, 'snr': 5.1},
#     ]
#
# detections = pipe.detect_with(my_detector)
# stats = pipe.analyze(match_radius=5.0)
# results = pipe.make_plots(plots=['completeness_1d'], show=True, save=True)